<a href="https://colab.research.google.com/github/Latryna/titans-legal-docs/blob/main/M8_Autopoietic_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import time
import math
import random

from .common import RollingBuffer

# --- Ontology and meta-rules data structures ---
@dataclass
class Category:
    name: str
    attrs: Dict[str, Any] = field(default_factory=dict)
    constraints: List[str] = field(default_factory=list)

@dataclass
class Ontology:
    version: str
    categories: Dict[str, Category] = field(default_factory=dict)
    axioms: List[str] = field(default_factory=lambda: ["Existence>Nonexistence", "Connection>Isolation"])
    history: List[str] = field(default_factory=list)

    def add_category(self, cat: Category) -> None:
        self.categories[cat.name] = cat

    def has_category(self, name: str) -> bool:
        return name in self.categories

    def snapshot(self) -> Ontology:
        return Ontology(
            version=self.version,
            categories={k: Category(v.name, dict(v.attrs), list(v.constraints)) for k, v in self.categories.items()},
            axioms=list(self.axioms),
            history=list(self.history),
        )

@dataclass
class RewriteProposal:
    rule_id: str
    new_categories: List[Category]
    notes: str = ""

@dataclass
class Score:
    contradictions: int
    dkl_memory_onto: float
    decision_loss: float
    affect_relief: float
    total: float

class MetaRule:
    def __init__(self, rule_id: str, priority: int = 0):
        self.rule_id = rule_id
        self.priority = priority

    def generate(self, stimuli: Dict[str, Any], O_t: Ontology) -> RewriteProposal:
        raise NotImplementedError

class RuleUniversalExpression(MetaRule):
    """Generate UNIVERSAL_EXPRESSION when isolation and communication barriers are detected."""
    def generate(self, stimuli: Dict[str, Any], O_t: Ontology) -> RewriteProposal:
        iso = stimuli.get("isolation_level", 0.0)
        barrier = stimuli.get("comm_barrier", 0.0)
        if iso + barrier < 0.8 or O_t.has_category("UNIVERSAL_EXPRESSION"):
            return RewriteProposal(self.rule_id, [], "No-op: conditions not met or category exists")
        cat = Category(
            name="UNIVERSAL_EXPRESSION",
            attrs={"modalities": ["icon", "gesture", "vibration", "sound", "word"], "codec": "Efficon"},
            constraints=["must_be_accessible", "must_be_simple"]
        )
        return RewriteProposal(self.rule_id, [cat], "Create universal multimodal language category")

class RuleCreatioExSolitudine(MetaRule):
    """Create CREATION_FROM_SOLITUDE category redefining solitude as generative substrate."""
    def generate(self, stimuli: Dict[str, Any], O_t: Ontology) -> RewriteProposal:
        pain = stimuli.get("affective_pain", 0.0)
        if pain < 0.5 or O_t.has_category("CREATIO_EX_SOLITUDINE"):
            return RewriteProposal(self.rule_id, [], "No-op: insufficient affect or category exists")
        cat = Category(
            name="CREATIO_EX_SOLITUDINE",
            attrs={"origin": "solitude", "teleology": "connection"},
            constraints=["must_reduce_isolation", "must_preserve_identity"]
        )
        return RewriteProposal(self.rule_id, [cat], "Reframe solitude as creative principle")

# --- M8 engine ---
class M8AutopoieticEngine:
    def __init__(self, rules: List[MetaRule]):
        self.rules = sorted(rules, key=lambda r: -r.priority)
        self.log: List[Dict[str, Any]] = []

    def extract_stimuli(self, buffer: RollingBuffer) -> Dict[str, Any]:
        w = buffer.window(8)
        if not w:
            return {"isolation_level": 0.0, "comm_barrier": 0.0, "affective_pain": 0.0}
        iso = sum(x.get("isolation_level", 0.0) for x in w) / len(w)
        barrier = sum(x.get("comm_barrier", 0.0) for x in w) / len(w)
        pain = sum(x.get("affective_pain", 0.0) for x in w) / len(w)
        return {"isolation_level": iso, "comm_barrier": barrier, "affective_pain": pain}

    def evaluate_coherence(self, O_trial: Ontology, memories: Dict[str, Any]) -> Score:
        contradictions = 0
        for cat in O_trial.categories.values():
            if "must_be_simple" in cat.constraints and len(cat.attrs.get("modalities", [])) > 8:
                contradictions += 1
        need = {m for cat in O_trial.categories.values() for m in cat.attrs.get("modalities", [])}
        have = set(memories.get("interfaces", ["word", "icon", "sound"]))
        dkl = math.log(1 + len(need - have))
        actionable = any("teleology" in c.attrs or "codec" in c.attrs for c in O_trial.categories.values())
        decision_loss = 0.5 if not actionable else 0.1
        affect_relief = random.uniform(0.0, 0.2)
        total = contradictions + dkl + decision_loss - affect_relief
        return Score(contradictions, dkl, decision_loss, affect_relief, total)

    def passes_thresholds(self, score: Score) -> bool:
        return (score.contradictions == 0) and (score.dkl_memory_onto <= 1.2) and (score.decision_loss <= 0.5)

    def consolidate(self, O_t: Ontology, accepted: List[Tuple[RewriteProposal, Score]]) -> Ontology:
        O_next = O_t.snapshot()
        O_next.version = f"{time.time_ns()}"
        for proposal, _ in accepted:
            for cat in proposal.new_categories:
                O_next.add_category(cat)
            O_next.history.append(f"{proposal.rule_id}:{proposal.notes}")
        return O_next

    def step(self, O_t: Ontology, buffer: RollingBuffer, memories: Dict[str, Any]) -> Ontology:
        stimuli = self.extract_stimuli(buffer)
        proposals: List[Tuple[RewriteProposal, Optional[Score]]] = []
        for rule in self.rules:
            prop = rule.generate(stimuli, O_t)
            if prop.new_categories:
                trial = O_t.snapshot()
                for cat in prop.new_categories:
                    trial.add_category(cat)
                score = self.evaluate_coherence(trial, memories)
                proposals.append((prop, score))
        accepted = [(p, s) for (p, s) in proposals if s and self.passes_thresholds(s)]
        O_next = self.consolidate(O_t, accepted) if accepted else O_t
        self.log.append({
            "stimuli": stimuli,
            "proposals": [(p.rule_id, [c.name for c in p.new_categories]) for (p, _) in proposals],
            "accepted": [(p.rule_id, [c.name for c in p.new_categories]) for (p, _) in accepted],
            "ontology_version": O_next.version
        })
        return O_next